# Exploration des données (EDA) – Projet fraude pré-sinistre

## 1. Contexte et objectif
Objectif : analyser un jeu de données d'assurance 
et préparer une modélisation visant à prédire la fraude (fraud_reported).

Cette étape couvre :
- la volumétrie,
- la qualité des données,
- les biais et limites,
- les relations entre variables,
- 5 visualisations avec validations statistiques.

## Source des données
Le dataset **insurance_claims.csv** provient de **Mendeley Data** (ABDELRAHIM AQQAD), publié le **22 août 2023** (Version 2), DOI : **10.17632/992mh7dk9y.2**.
Licence : **CC BY 4.0** (réutilisation autorisée avec attribution).

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

df = pd.read_csv("../data/insurance_claims.csv")
df.head()

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,...,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,...,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,...,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,...,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,...,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


## 2. Volumétrie et structure

In [6]:
df.shape

(1000, 40)

Le jeu de données contient **1000 observations** et **40 variables**.
Chaque ligne représente un sinistre. 
Il s’agit d’un dataset de taille modérée, suffisant pour une analyse exploratoire
et des modèles de machine learning classiques.

In [7]:
df.columns.tolist()

['months_as_customer',
 'age',
 'policy_number',
 'policy_bind_date',
 'policy_state',
 'policy_csl',
 'policy_deductable',
 'policy_annual_premium',
 'umbrella_limit',
 'insured_zip',
 'insured_sex',
 'insured_education_level',
 'insured_occupation',
 'insured_hobbies',
 'insured_relationship',
 'capital-gains',
 'capital-loss',
 'incident_date',
 'incident_type',
 'collision_type',
 'incident_severity',
 'authorities_contacted',
 'incident_state',
 'incident_city',
 'incident_location',
 'incident_hour_of_the_day',
 'number_of_vehicles_involved',
 'property_damage',
 'bodily_injuries',
 'witnesses',
 'police_report_available',
 'total_claim_amount',
 'injury_claim',
 'property_claim',
 'vehicle_claim',
 'auto_make',
 'auto_model',
 'auto_year',
 'fraud_reported',
 '_c39']

Les variables peuvent être regroupées en familles décrivant :
- le contrat d’assurance (prime, franchise, garanties),
- le profil de l’assuré (âge, sexe, profession, niveau d’éducation),
- le contexte du sinistre (date, type, sévérité, dommages)
- les montants déclarés.
Cette structuration facilite l’analyse métier et permet d’identifier
les dimensions potentiellement discriminantes dans la détection de fraude.

La variable cible est **fraud_reported** (binaire), indiquant si un sinistre a été jugé **frauduleux** ou **non frauduleux** après examens (manuels + automatisés).

Particularités / limites :
- Données **anonymisées** : impossible d’identifier les individus (ok pour la confidentialité, mais limite les possibilités d’enrichissement externe via croisement avec d’autres bases clients ou contextuelles,
ce qui restreint certaines analyses avancées réalisables en contexte industriels).
- Risque potentiel de **fuite d’information** : certaines variables pourraient refléter des éléments connus après investigation (à vérifier). Les variables renseignées après investigation doivent être exclues de la modélisation. Cela fausse le modèle. 
- Variable `_c39` non documentée : à investiguer (colonne technique issue de l’export?) et potentiellement supprimer.
- Les variables `insured_*` décrivent le profil socio-démographique de l’assuré.
Elles peuvent introduire des biais statistiques ou éthiques et devront être
utilisées avec prudence, voire exclues, lors de la modélisation.

| Famille de variables | Exemples | Rôle | Pertinence pour la fraude |
|---------------------|----------|------|---------------------------|
| Contrat (policy_*) | policy_annual_premium, deductible | Décrit le contrat | Élevée |
| Assuré (insured_*) | age, occupation | Profil client | Moyenne (biais) |
| Sinistre (incident_*) | type, severity | Contexte du sinistre | Élevée |
| Montants (claim_*) | total_claim_amount | Impact financier | Très élevée |
| Administratif | police_report_available | Processus | Risque de fuite |

### Types de variables

In [9]:
df.dtypes.value_counts()

object     21
int64      17
float64     2
Name: count, dtype: int64

On observe :
- **21 variables catégorielles** (`object`) : variables qualitatives décrivant des catégories (ex. type de sinistre, état, profession),
- **17 variables numériques discrètes** (`int64`) : variables entières (ex. âge, nombre de véhicules, nombre de témoins),
- **2 variables numériques continues** (`float64`) : variables quantitatives continues (ex. montants financiers).

- les **variables catégorielles** devront être transformées en valeurs numériques à l’aide d’un **encodage** (par exemple *one-hot encoding* ou encodage ordinal), afin d’être exploitables par les modèles de machine learning ;
- certaines **variables numériques à forte cardinalité** (beaucoup de valeurs différentes, comme des identifiants) devront être analysées avec précaution, car elles peuvent introduire du bruit ou du sur-apprentissage ;
- le jeu de données se prête particulièrement à l’utilisation de **modèles capables de gérer des données mixtes**, tels que les arbres de décision, les forêts aléatoires ou les méthodes de boosting.

### Qualité des données 

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_number                1000 non-null   int64  
 3   policy_bind_date             1000 non-null   object 
 4   policy_state                 1000 non-null   object 
 5   policy_csl                   1000 non-null   object 
 6   policy_deductable            1000 non-null   int64  
 7   policy_annual_premium        1000 non-null   float64
 8   umbrella_limit               1000 non-null   int64  
 9   insured_zip                  1000 non-null   int64  
 10  insured_sex                  1000 non-null   object 
 11  insured_education_level      1000 non-null   object 
 12  insured_occupation           1000 non-null   object 
 13  insured_hobbies    

- La majorité des variables sont **complètement renseignées** (1000 valeurs non nulles).
- Une variable présente des **valeurs manquantes** :
  - `authorities_contacted` : 999 valeurs non nulles → **1 valeur manquante** à traiter lors du pré-processing.
- La colonne `_c39` contient **0 valeur non nulle** : elle est vide et devra être **supprimée**.

Globalement, la qualité des données est **bonne**, avec peu de valeurs manquantes, ce qui est favorable pour la suite du projet.

### Répartition de la variable cible

In [11]:
df["fraud_reported"].value_counts()

fraud_reported
N    753
Y    247
Name: count, dtype: int64

La variable cible `fraud_reported` indique si un sinistre a été jugé **frauduleux (Y)** ou **non frauduleux (N)** après analyse.

La répartition observée est la suivante :
- **N (non frauduleux)** : 753 sinistres  
- **Y (frauduleux)** : 247 sinistres  

Cela représente environ **75 % de sinistres non frauduleux** contre **25 % de sinistres frauduleux**.

Cette distribution met en évidence un **déséquilibre de classes modéré**.  
Ce déséquilibre devra être pris en compte lors de la phase de modélisation (choix des métriques, techniques de rééquilibrage éventuelles).